In [45]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import PolynomialFeatures

In [46]:
df = pd.read_csv('data/merged/train_processed.csv')
df.describe()

,pickup_longitude_rounded,pickup_latitude_rounded,dropoff_longitude_rounded,dropoff_latitude_rounded,passenger_count,trip_duration,fare_amount
count,126473.000000,126473.000000,126473.000000,126473.000000,126473.000000,1.264730e+05,126473.000000
mean,-73.976081,40.753639,-73.977687,40.754277,1.183850,7.674804e+02,9.233110
std,0.031296,0.022355,0.027004,0.022086,0.671001,5.578705e+03,8.339225
min,-74.194000,40.616000,-74.194000,40.602000,1.000000,1.000000e+00,-5.500000
25%,-73.991000,40.742000,-73.991000,40.742000,1.000000,3.310000e+02,5.300000
50%,-73.982000,40.756000,-73.981000,40.756000,1.000000,5.240000e+02,6.900000
75%,-73.969000,40.768000,-73.970000,40.768000,1.000000,8.280000e+02,9.500000
max,-73.422000,41.022000,-73.422000,41.022000,6.000000,1.763934e+06,152.450000


# Fare Price

In [31]:
# Linear Regression
X = df.drop(columns=['trip_duration', 'fare_amount'])
y = df['fare_amount']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print('Train MSE:', mean_squared_error(y_train, model.predict(X_train)))
print('Test MSE:', mean_squared_error(y_test, model.predict(X_test)))

Train MSE: 40.04639195596812
Test MSE: 44.96676694989141


In [60]:
# Polynomial Regression
X = df.drop(columns=['trip_duration', 'fare_amount'])
y = df['fare_amount']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

poly = PolynomialFeatures(degree=2)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.fit_transform(X_test)

model = LinearRegression()
model.fit(X_train_poly, y_train)

print('Train MSE:', mean_squared_error(y_train, model.predict(X_train_poly)))
print('Test MSE:', mean_squared_error(y_test, model.predict(X_test_poly)))

Train MSE: 14.494604487097238
Test MSE: 15.001427019371231


In [61]:
from sklearn.metrics import r2_score
r2 = r2_score(y_test, model.predict(X_test_poly))
print(f'R-squared: {r2}')

R-squared: 0.7948199600175319


# Duration

In [62]:
# Linear Regression
X = df.drop(columns=['trip_duration', 'fare_amount'])
y = df['trip_duration']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print('Train MSE:', mean_squared_error(y_train, model.predict(X_train)))
print('Test MSE:', mean_squared_error(y_test, model.predict(X_test)))

Train MSE: 6076367.035119523
Test MSE: 130716325.9973601


In [63]:
# Polynomial Regression
X = df.drop(columns=['trip_duration', 'fare_amount'])
y = df['trip_duration']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

poly = PolynomialFeatures(degree=5)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.fit_transform(X_test)

model = LinearRegression()
model.fit(X_train_poly, y_train)

print('Train MSE:', mean_squared_error(y_train, model.predict(X_train_poly)))
print('Test MSE:', mean_squared_error(y_test, model.predict(X_test_poly)))

Train MSE: 5946989.4007397825
Test MSE: 130475851.49982373


In [64]:
r2 = r2_score(y_test, model.predict(X_test_poly))
print(f'R-squared: {r2}')

R-squared: 0.004228180352492172


In [29]:
y_pred = model.predict(X_test)

print(pd.DataFrame(y_pred - y_test).describe())

        fare_amount
count  25295.000000
mean      -0.052131
std        6.705656
min     -126.322455
25%       -1.723568
50%        0.886106
75%        3.255775
max       63.481494


In [65]:
df.columns

Index(['pickup_longitude_rounded', 'pickup_latitude_rounded',
       'dropoff_longitude_rounded', 'dropoff_latitude_rounded',
       'passenger_count', 'trip_duration', 'fare_amount'],
      dtype='object')

In [23]:
import folium
nyc_center = [40.7128, -74.0060]

nyc_map = folium.Map(location=nyc_center, zoom_start=12)

for _, row in df.iterrows():
    folium.CircleMarker(
        location=[row['pickup_latitude_rounded'], row['pickup_longitude_rounded']],
        radius=1,
        color='blue',
        fill=True,
        fill_opacity=0.4
    ).add_to(nyc_map)

for _, row in df.iterrows():
    folium.CircleMarker(
        location=[row['dropoff_latitude_rounded'], row['dropoff_longitude_rounded']],
        radius=1,
        color='red',
        fill=True,
        fill_opacity=0.4
    ).add_to(nyc_map)

# Save map to an HTML file and display it
nyc_map.save('nyc_pickup_dropoff_map.html')